# Lab 4 · Entrenamiento en la nube (Colab / Kaggle)

**Siempre se usa `Run all`.** Lo único que cambias es `MODE` en la celda 1:

| `MODE` | Qué hace | Cuándo |
|---|---|---|
| `"calibrar"` | Instala y mide la velocidad en esta máquina (~5 min) y **se detiene** | La primera vez en cada tipo de GPU |
| `"entrenar"` | Entrena, exporta el agente, lo verifica con `evaluar.py` y muestra el reporte | Corrida real. Si la sesión se corta: `Run all` otra vez y **reanuda solo** |
| `"exportar"` | **No entrena.** Empaqueta el mejor agente de un run que ya existe (mismo `CONFIG`/`OVERRIDES name=`), lo verifica con `evaluar.py` y deja el zip | Para sacar el modelo mientras otra sesión sigue entrenando, o si se acabó el tiempo |

Colab: *Entorno de ejecución → Cambiar tipo* (T4 + RAM alta para SAC). Los runs se guardan en tu Drive (`lab4_runs/`).
Kaggle: activa *Internet* en Settings; los runs quedan en `/kaggle/working/runs` (usa *Save Version → Run All*).

In [ ]:
#@title 1 · Parámetros (lo único que se toca)
MODE = "calibrar"     # "calibrar" | "entrenar" | "exportar" (empaqueta el mejor agente sin entrenar)
REPO_URL = "https://github.com/Vasco2510/walker.git"
BRANCH = "main"
CONFIG = "configs/sac_max.yaml"   # PPO en CPU: "configs/ppo_curriculum.yaml"
OVERRIDES = []        # p.ej. ["name=sac_max_s1", "seed=1"]  (segunda semilla en otra sesión)
FRESH = False         # True = empezar de cero (el run anterior se ARCHIVA, no se borra)
SECRET_NAME = None    # repo privado: nombre del secret con tu token de GitHub

import sys
PY = f'"{sys.executable}"'   # ruta de Python entrecomillada (funciona aunque tenga espacios)

In [ ]:
#@title 2 · Bootstrap: clonar/actualizar repo e instalar dependencias
import os, subprocess, sys
from pathlib import Path

def sh(cmd, cwd=None):
    """Ejecuta mostrando la salida en vivo; devuelve las líneas."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line.rstrip())
    if p.wait() != 0:
        raise RuntimeError(f"Falló: {cmd}")
    return lines

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()
ON_COLAB = not ON_KAGGLE and "COLAB_RELEASE_TAG" in os.environ   # Kaggle también define variables de Colab: se mira Kaggle primero

if Path("training").is_dir() and Path("walker_lab").is_dir():      # ya estamos dentro del repo (uso local)
    REPO = Path.cwd()
else:
    url = REPO_URL
    if SECRET_NAME:
        if ON_COLAB:
            from google.colab import userdata; token = userdata.get(SECRET_NAME)
        else:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret(SECRET_NAME)
        url = url.replace("https://", f"https://{token}@")
    REPO = Path("/kaggle/working/repo" if ON_KAGGLE else "/content/repo")
    if (REPO / ".git").is_dir():
        sh(f"git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD", cwd=REPO)
    else:
        sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")

if ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")   # los runs persisten aquí

os.chdir(REPO)
sh("pip install -q swig && pip install -q -r requirements-train.txt")   # swig por si Box2D debe compilarse
sh("git log -1 --oneline")

In [ ]:
#@title 2b · Calibración (solo si MODE = "calibrar")
if MODE == "calibrar":
    sh(f"{PY} -u -m training.benchmark")
    print("\n>>> Copia la tabla de arriba y pásasela a quien te ayuda. Luego cambia MODE a 'entrenar'.")
else:
    print("Calibración omitida (MODE =", MODE, ")")

In [ ]:
#@title 3 · Entrenar ("entrenar", reanuda solo) o localizar el run ("exportar")
if MODE == "entrenar":
    args = f"--config {CONFIG} " + " ".join(f"--set {o}" for o in OVERRIDES) + (" --fresh" if FRESH else "")
    out = sh(f"{PY} -u -m training.train {args}")
    RUN_DIR = Path(next(l for l in reversed(out) if l.startswith("RUN_DIR=")).split("=", 1)[1])
elif MODE == "exportar":
    sys.path.insert(0, os.getcwd())   # el repo (ya estamos dentro tras la celda 2)
    from training.runtime import runs_root
    names = [o.split("=", 1)[1] for o in OVERRIDES if o.startswith("name=")]
    RUN_DIR = runs_root() / (names[0] if names else Path(CONFIG).stem)
    print("Exportando el mejor agente de:", RUN_DIR)
else:
    print("MODE = 'calibrar': no se entrena.")

In [ ]:
#@title 4 · Exportar agente, comprobar con evaluar.py y empaquetar (MODE "entrenar" o "exportar")
if MODE in ("entrenar", "exportar"):
    import shutil
    sh(f'{PY} -m training.export "{RUN_DIR}" --to agente')                      # best_agent -> agente/
    sh(f"{PY} evaluar.py --scenario escenarios/validacion/maraton_01.json")  # requisito de la entrega
    print((RUN_DIR / "best_agent" / "meta.json").read_text()[:700])
    for d in Path("agente").rglob("__pycache__"):
        shutil.rmtree(d, ignore_errors=True)   # el ZIP no debe llevar caché de Python
    zip_path = shutil.make_archive(str(RUN_DIR / "agente"), "zip", ".", "agente")        # el ZIP contiene la carpeta agente/
    print("Agente listo:", zip_path)
    if ON_COLAB:
        from google.colab import files; files.download(zip_path)
    elif ON_KAGGLE:
        shutil.copy(zip_path, "/kaggle/working/agente.zip")   # descargable desde la pestaña Output

In [ ]:
#@title 5 · Reporte realista del run (solo si MODE = "entrenar"): velocidad, reparto de tiempo, ETA y evaluaciones
if MODE in ("entrenar", "exportar"):
    sh(f'{PY} -m training.report "{RUN_DIR}"')